# M1 SAR Ship Detection — Kaggle 2×T4 Benchmark

**Dataset:** HRSID — 4,042 train / 1,962 val · 800×800 SAR JPEG chips · single class: ship  
**Models:** YOLOv8m (2023 baseline) → YOLO11m-OBB (2024 oriented boxes) → YOLO26m (2026 Ultralytics flagship)  
**Hardware:** Kaggle 2×T4 (30 GB VRAM total), DDP, ~2 h/model → ~6 h total  


In [ ]:
%%capture
# Upgrade ultralytics to get YOLO26 support
!pip install -q -U ultralytics huggingface_hub wandb gdown
import ultralytics; print(f'ultralytics {ultralytics.__version__}')

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
from collections import defaultdict

WORK = Path('/kaggle/working')
REPO = WORK / 'internship'

# Clone the project repo
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/shaunmarv3/internship.git', str(REPO)],
        check=True
    )
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f'Repo: {REPO}  |  cwd: {os.getcwd()}')

In [ ]:
  import os, wandb

  os.environ['HF_TOKEN']      = ''
  os.environ['WANDB_API_KEY'] = ''

  wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
  print('Auth done')


In [ ]:
  import gdown, zipfile

  RAW = WORK / 'hrsid_raw'
  RAW.mkdir(exist_ok=True)

  MAIN_ZIP = RAW / 'hrsid.zip'
  NEG_ZIP  = RAW / 'hrsid_neg.zip'

  if not MAIN_ZIP.exists():
      print('Downloading main (~614 MB)...')
      gdown.download(id='1NY3ovgc-woDlNoQdyqzRB3t9McOBH5Ms', output=str(MAIN_ZIP), quiet=False)

  if not NEG_ZIP.exists():
      print('Downloading negatives (~220 MB)...')
      gdown.download(id='1U0Sj1SHoq-2VjXXUKwpXae6rBI3YjyDP', output=str(NEG_ZIP), quiet=False)

  EXTRACT = RAW / 'extracted'
  EXTRACT.mkdir(exist_ok=True)
  NEG_EXTRACT = RAW / 'negatives'
  NEG_EXTRACT.mkdir(exist_ok=True)

  if not any(EXTRACT.rglob('*.jpg')):
      print('Extracting main zip...')
      with zipfile.ZipFile(MAIN_ZIP) as zf:
          zf.extractall(EXTRACT)

  if not any(NEG_EXTRACT.rglob('*.png')):
      print('Extracting negatives...')
      with zipfile.ZipFile(NEG_ZIP) as zf:
          zf.extractall(NEG_EXTRACT)

  train_json  = next(EXTRACT.rglob('train2017.json'))
  val_json    = next(EXTRACT.rglob('test2017.json'))
  img_root    = next(EXTRACT.rglob('*.jpg')).parent
  neg_pngs    = sorted(NEG_EXTRACT.rglob('*.png'))

  print(f'Train JSON : {train_json}')
  print(f'Val JSON   : {val_json}')
  print(f'Images     : {len(list(img_root.glob("*.jpg")))} jpgs in {img_root}')
  print(f'Negatives  : {len(neg_pngs)} pngs')


In [ ]:
  def coco_to_yolo(coco_json_path, src_img_dir, dst_img_dir, dst_lbl_dir):
      with open(coco_json_path) as f:
          coco = json.load(f)
      dst_img_dir = Path(dst_img_dir); dst_img_dir.mkdir(parents=True, exist_ok=True)
      dst_lbl_dir = Path(dst_lbl_dir); dst_lbl_dir.mkdir(parents=True, exist_ok=True)
      src_img_dir = Path(src_img_dir)
      id2img = {img['id']: img for img in coco['images']}
      ann_map = defaultdict(list)
      for ann in coco['annotations']:
          ann_map[ann['image_id']].append(ann)
      for img_id, info in id2img.items():
          W, H = info['width'], info['height']
          stem = Path(info['file_name']).stem
          src  = src_img_dir / info['file_name']
          dst_img = dst_img_dir / info['file_name']
          if not dst_img.exists() and src.exists():
              shutil.copy2(src, dst_img)
          lines = []
          for ann in ann_map[img_id]:
              x, y, w, h = ann['bbox']
              cx, cy = (x + w/2)/W, (y + h/2)/H
              lines.append(f'0 {cx:.6f} {cy:.6f} {w/W:.6f} {h/H:.6f}')
          (dst_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))
      print(f'  {len(id2img)} images/labels done')

  YOLO_DIR = WORK / 'HRSID_yolo'
  print('Converting train...'); coco_to_yolo(train_json, img_root, YOLO_DIR/'images'/'train', YOLO_DIR/'labels'/'train')
  print('Converting val...');   coco_to_yolo(val_json,   img_root, YOLO_DIR/'images'/'val',   YOLO_DIR/'labels'/'val')

  # Add 400 background negatives
  COLLISION = 'P0128_600_1400_4800_5600'
  for png in neg_pngs:
      stem = png.stem + ('_neg' if png.stem == COLLISION else '')
      dst = YOLO_DIR / 'images' / 'train' / (stem + '.png')
      if not dst.exists(): shutil.copy2(png, dst)
      lbl = YOLO_DIR / 'labels' / 'train' / (stem + '.txt')
      if not lbl.exists(): lbl.write_text('')

  n_train = len(list((YOLO_DIR/'images'/'train').glob('*')))
  n_val   = len(list((YOLO_DIR/'images'/'val').glob('*')))
  print(f'Train: {n_train}  Val: {n_val}')   # expect 4042 / 1962

  # data.yaml
  (YOLO_DIR / 'data.yaml').write_text(
  f"""path: {YOLO_DIR}
  train: images/train
  val:   images/val
  nc: 1
  names: ['ship']
  """)
  print('data.yaml written')


In [ ]:
  import cv2, numpy as np

  def convert_to_obb(coco_json_path, out_lbl_dir):
      with open(coco_json_path) as f:
          coco = json.load(f)
      out_lbl_dir = Path(out_lbl_dir); out_lbl_dir.mkdir(parents=True, exist_ok=True)
      id2img = {img['id']: img for img in coco['images']}
      ann_map = defaultdict(list)
      for ann in coco['annotations']:
          if ann.get('segmentation'): ann_map[ann['image_id']].append(ann)
      for img_id, info in id2img.items():
          W, H = info['width'], info['height']
          stem = Path(info['file_name']).stem
          lines = []
          for ann in ann_map[img_id]:
              for seg in ann['segmentation']:
                  if len(seg) < 6: continue
                  pts = np.array(seg, dtype=np.float32).reshape(-1, 2)
                  corners = cv2.boxPoints(cv2.minAreaRect(pts))
                  norm = corners / np.array([W, H])
                  lines.append('0 ' + ' '.join(f'{v:.6f}' for v in norm.flatten()))
          (out_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))
      print(f'  {len(id2img)} OBB label files written')

  OBB_DIR = WORK / 'HRSID_obb'
  for split in ['train', 'val']:
      lnk = OBB_DIR / 'images' / split
      lnk.mkdir(parents=True, exist_ok=True)
      # symlink images from YOLO_DIR to save disk
      if not (lnk / 'symlinked').exists():
          for f in (YOLO_DIR/'images'/split).glob('*'):
              t = lnk / f.name
              if not t.exists(): os.symlink(f.resolve(), t)
          (lnk / 'symlinked').touch()

  print('OBB train labels...'); convert_to_obb(train_json, OBB_DIR/'labels'/'train')
  print('OBB val labels...');   convert_to_obb(val_json,   OBB_DIR/'labels'/'val')

  # OBB negatives — same empty labels
  for png in neg_pngs:
      stem = png.stem + ('_neg' if png.stem == COLLISION else '')
      lbl = OBB_DIR / 'labels' / 'train' / (stem + '.txt')
      if not lbl.exists(): lbl.write_text('')

  (OBB_DIR / 'data.yaml').write_text(
  f"""path: {OBB_DIR}
  train: images/train
  val:   images/val
  nc: 1
  names: ['ship']
  """)
  print('OBB data.yaml written')


In [ ]:
  for name, d in [('YOLO', YOLO_DIR), ('OBB', OBB_DIR)]:
      for s in ['train', 'val']:
          imgs = len(list((Path(d)/'images'/s).glob('*')))
          lbls = len(list((Path(d)/'labels'/s).glob('*.txt')))
          print(f'{name} {s}: {imgs} images, {lbls} labels')


In [ ]:
  for split in ['train', 'val']:
      sentinel = OBB_DIR / 'images' / split / 'symlinked'
      if sentinel.exists():
          sentinel.unlink()
          print(f'Removed sentinel: {sentinel}')

  # Verify again
  for name, d in [('YOLO', YOLO_DIR), ('OBB', OBB_DIR)]:
      for s in ['train', 'val']:
          imgs = len([f for f in (Path(d)/'images'/s).iterdir() if f.suffix in ('.jpg','.png','.tif')])
          lbls = len(list((Path(d)/'labels'/s).glob('*.txt')))
          print(f'{name} {s}: {imgs} images, {lbls} labels')


In [ ]:
  os.chdir(REPO)
  !python src/models/train_detection.py \
      --model    yolov8m \
      --data     {YOLO_DIR}/data.yaml \
      --project  {WORK}/checkpoints/vessel \
      --device   0,1 \
      --push_hf \
      --wandb_project maritime-vessel


In [ ]:
  os.chdir(REPO)
  !python src/models/train_detection.py \
      --model    yolo11m-obb \
      --data     {OBB_DIR}/data.yaml \
      --project  {WORK}/checkpoints/vessel \
      --device   0,1 \
      --push_hf \
      --wandb_project maritime-vessel


In [ ]:
  os.chdir(REPO)
  !python src/models/train_detection.py \
      --model    yolo26m \
      --data     {YOLO_DIR}/data.yaml \
      --project  {WORK}/checkpoints/vessel \
      --device   0,1 \
      --push_hf \
      --wandb_project maritime-vessel


In [ ]:
  import pandas as pd

  ckpt_base = WORK / 'checkpoints' / 'vessel'
  models = [
      ('YOLOv8m',      'hrsid_yolov8m'),
      ('YOLO11m-OBB',  'hrsid_yolo11m_obb'),
      ('YOLO26m',      'hrsid_yolo26m'),
  ]
  rows = []
  for label, run in models:
      csv = ckpt_base / run / 'results.csv'
      if not csv.exists():
          rows.append({'Model': label, 'mAP50': 'pending', 'mAP50-95': 'pending', 'Precision': 'pending', 'Recall': 'pending'})
          continue
      r = pd.read_csv(csv).iloc[-1]
      rows.append({
          'Model':     label,
          'mAP50':     f"{r.get('metrics/mAP50(B)', r.get('metrics/mAP50', 0)):.4f}",
          'mAP50-95':  f"{r.get('metrics/mAP50-95(B)', r.get('metrics/mAP50-95', 0)):.4f}",
          'Precision': f"{r.get('metrics/precision(B)', r.get('metrics/precision', 0)):.4f}",
          'Recall':    f"{r.get('metrics/recall(B)', r.get('metrics/recall', 0)):.4f}",
      })
  print(pd.DataFrame(rows).to_string(index=False))
